# Wildfire Prediction

## Table of content

- Read data

- Exploratory data analysis
    - Class balance
    - FWI histograms
    - Missing values

- Feature engineering
    - Train/test split

- Train models
    - Random Forest

- Evaluate
    - ROC
    - confusion matrix
    - feature importance

## Read data

### Libraries

In [12]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import re

from scripts.s00_set_parameters import PARAMETERS
from ml_models.ml_utils import train_test_temporal_split
from typing import Any


### Load data

In [ ]:

import os 
# Base path/location of data
path_base = PARAMETERS['DATA_DIR']/"MLInputs"
# Input files to load
files_to_load = {"resnet_default_weights":  "2026-07-12_ml_input.csv",
                 "resnet_layer4_finetuned": "2026-07-19_ml_input_layer4_finetuned.csv"}
# Generate full paths 
files_to_load = {k: os.path.join(path_base, v) for (k, v) in files_to_load.items()}
# Spefic data types requirements
string_cols = {"composite_key": "string", "composite_key_y": "string", "bridge_composite_key_y": "string"}
date_cols   = ["date_y", "date"]
# Load data 
df_resnet_default   = pd.read_csv(files_to_load["resnet_default_weights"], dtype  = string_cols, parse_dates=date_cols) #type: ignore
df_resnet_finetuned = pd.read_csv(files_to_load["resnet_layer4_finetuned"], dtype = string_cols, parse_dates=date_cols) #type: ignore


## Exploratory data analysis

### Data structure

Ensure that the structure of both data frames is the same. The only difference should be the values in the feature columns - everything else should be equal

In [ ]:
# Validate structure is the same
print("Structure Validation: ")
print(f"- Columns match:    {df_resnet_finetuned.columns.equals(df_resnet_default.columns)}")
print(f"- Data types match: {df_resnet_finetuned.dtypes.equals(df_resnet_default.dtypes)}")
print(f"- Data Shape match: {df_resnet_finetuned.shape == df_resnet_default.shape}")

# Validate that the only data differences are found in feature columns, all other data points should be the same between data frames 
feat_cols               = [c for c in df_resnet_default.columns if re.search("^feat_", c)]
df_minus_feat_default   = df_resnet_default.drop(columns=feat_cols)
df_minus_feat_finetuned = df_resnet_finetuned.drop(columns=feat_cols)
# Validate data matches
print("\nData Validation:")
try:
    pd.testing.assert_frame_equal(df_minus_feat_default,  df_minus_feat_finetuned)
    print("- Non-feature data match: True")
except AssertionError as e:
    print("- Non-feature data match: False")
    print(e)

Structure Validation: 
- Columns match:    True
- Data types match: True
- Data Shape match: True

Data Validation:
- Non-feature data match: True


The full dataset contains the columns below. All the columns that end with suffix `y` refer to be values to be predicted. These are the values that the model needs to predict. The rest of the columns contain the data to train the model. The training data (non `y` columns) are from a t-1 from value to predict. 

Structure Validation: 
- Columns match:    True
- Data types match: True
- Data Shape match: True


In [ ]:
# Check assumptions for all datasets
for k, v in files.items():
    for _, row in v.iterrows():
        row: Any
        days_diff = (row.date_y - row.date).days
        if days_diff != 1:
            print('❌ Date assumption of t-1 not met!')
    print(f"✅ t-1 assumption validated for dataset [{k}]\n  [{df_ml.shape[0]} rows checked]")

### Class balance

The sampling procedure implemented a 2:1 no-fire to fire ratio when selecting the samples. The reason of this is that no-fire events are much more common than fire events. However, using the real distribution would have produced an extremely imbalanced dataset. To mitigate this, a 2:1 ratio was implemented to maintain the frequency property but reducing the class imbalance to a more manageable state, as done by previous studies. 

The complete details of the sampling procedure are in:
- Functions: `src/sampling/sampling_functions.py` 
- Pipeline: `src/pipelines/sampling_pipeline.py`

In [ ]:
for k, v in files.items():
    print(f"\n=== DataSet: {k} ===")
    print(v['fire_lbl_y'].value_counts(normalize=True))

### FWI Distribution

The Fire Weather Index data does not vary accross data sets loaded. The only change is applied to Sentinel2, therefore, this section uses the preset `df_ml` to conduct the data exploration tasks.

The Fire Weather Index is used as a predictor variable in the model. The histogram below compares the distribution of FWI values for fire and no-fire classes.

The class imbalance in the predictor variable is evident, with only 1,816 fire observations vs 36,388 no-fire observations.

However, the descriptive statistics show that the fire class has consistently higher FWI values than the no fire, with the median of fire observations almost 3 times higher than the no-fire counterpart. It is worth nothing, however, that the max value of no-fire is greater than the fire (more than 2sd above the max). This is the case for 7/36388 of the no-fire observations, which indicates these are rate events

In [ ]:
df_fwi_descriptives = df_ml[['fwi_mean', 'fire_lbl']].groupby('fire_lbl').describe()
df_fwi_descriptives

In [ ]:
# Count of no fire observations above the max of Fire
fwi_fire_lbl_max = df_fwi_descriptives.loc[True, ("fwi_mean", "max")]
df_ml[ (df_ml['fire_lbl'] == False) & (df_ml['fwi_mean'] > fwi_fire_lbl_max)].shape[0]

The histogram shows a greater concentration of no fire observations for FWI values close to zero. As FWI increases, the concentration of fire observations becomes greater than no fire 

In [ ]:

sns.histplot(data=df_ml, 
             x= 'fwi_mean', 
             hue = 'fire_lbl', 
             bins = 40, 
             stat = 'density', 
             common_norm = False,
             alpha = 0.5,
             palette = {False:"#5CC6F0",
                        True: "#F54125" },
              edgecolor = 'white')
plt.title("Distribution of FWI by Fire Class")
plt.xlabel("Fire Weatehr Index (FWI)")
plt.ylabel("Density")

plt.xlim(0, df_ml['fwi_mean'].max())

ax = plt.gca()
legend = ax.get_legend()
if legend is not None:
    legend.set_title("Fire Label")
    legend.texts[0].set_text("No Fire")
    legend.texts[1].set_text("Fire")

In [ ]:
sns.kdeplot(
    data=df_ml,
    x="fwi_mean",
    hue="fire_lbl",
    fill = True,
    common_norm=False,
    alpha=0.3,
    cut = 0,
    clip = (0, None)
)

## Feature engineering

### Train/Test split

In [5]:
for k, v in files.items():
    print(f"\n=== DataSet: {k} ===") 
    year_dict = {}
    for row in v.itertuples():
        row: Any
        year_str = str(row.date.year)
        year_dict[year_str] = year_dict.get(year_str, 0) + 1

    print(f"Total rows {df_ml.shape[0]}")
    print(year_dict)


=== DataSet: resnet_default_weights ===
Total rows 38204
{'2018': 7337, '2019': 6695, '2020': 5202, '2021': 5612, '2022': 6165, '2023': 4065, '2024': 3128}

=== DataSet: resnet_layer4_finetuned ===
Total rows 38204
{'2018': 7337, '2019': 6695, '2020': 5202, '2021': 5612, '2022': 6165, '2023': 4065, '2024': 3128}


In [5]:

from sklearn.model_selection import TimeSeriesSplit

from sklearn.metrics import classification_report
from sklearn.metrics import f1_score
from sklearn.base import clone

def model_crossvalidation(model, X, y, model_name, n_splits = 8, verbose = False):
    tscv = TimeSeriesSplit(n_splits = n_splits)
    scores = []
    reports = []
    model_scores = {model_name: {}}

    for i, (train_index, test_index) in enumerate(tscv.split(X)):
        print(f"Model name: {model_name}. Fold {i}", end = "\r")
        # Split data based on cv fold
        X_train_cv = X.iloc[train_index]
        y_train_cv = y.iloc[train_index]

        X_test_cv = X.iloc[test_index]
        y_test_cv = y.iloc[test_index]

        # Create a fresh model for this fold
        model_cv = clone(model)
        # Train model
        model_cv.fit(X_train_cv, y_train_cv)
        # Generate predictions
        y_pred = model_cv.predict(X_test_cv)

        # Evaluate
        score = f1_score(y_test_cv, y_pred)
        scores.append(score)
        model_scores[model_name][f"F1 Score fold {i}"] = score
        report = classification_report(y_test_cv, y_pred, output_dict=True)
        reports.append(pd.DataFrame(report).T)
        if verbose:
            print(f"Fold {i}: {score:.3f}")
            print(classification_report(y_test_cv, y_pred))
            print(".....................................")
    mean_score = sum(scores) / len(scores)
    avg_report = (pd.concat(reports).groupby(level=0).mean())
    print(f"========== Model: {model_name} ==========\nMean F1 Score: {mean_score:.3f}\nTotal folds: {n_splits}\nAverage Class Report\n{avg_report}")
    model_scores[model_name]["F1 Mean Score"] = mean_score
    return model_scores


In [6]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

random_state = 42
class_weight = 'balanced'
max_iter = 1000

# Initialise models
models = {'random_forest': RandomForestClassifier(random_state=random_state, class_weight=class_weight),
          'svm':           Pipeline([("scaler", StandardScaler()),
                                     ("svm", LinearSVC(C=1.0,
                                                       class_weight=class_weight,
                                                       random_state=random_state,
                                                        max_iter = max_iter)) ]),
            'lr':          Pipeline([("scaler", StandardScaler()),
                                     ("lr", LogisticRegression(class_weight=class_weight,
                                                               random_state=random_state,
                                                               max_iter = max_iter)) ])}


In [7]:
# Run FWI as this doesnt change across datasets
verbose_as = False
df_train, df_test = train_test_temporal_split(df_ml)
#df_ml = df_ml.sort_values("date").reset_index(drop=True)

X = df_train[["fwi_mean"]]
y = df_train["fire_lbl_y"]
# y_train = df_train.loc[:, 'fire_lbl_y']
# y_test = df_test.loc[:, 'fire_lbl_y']

# X_train_fwi = df_train.loc[:, ['fwi_mean']]
# X_test_fwi  =  df_test.loc[:, ['fwi_mean']]

rf_fwi = model_crossvalidation(models['random_forest'], X, y, 'random_forest_fwi', n_splits = 8, verbose = verbose_as)
lr_fwi = model_crossvalidation(models['lr'], X, y, 'logistic_regression_fwi', n_splits = 8, verbose = verbose_as)
lr_fwi = model_crossvalidation(models['svm'], X, y, 'svm_fwi', n_splits = 8, verbose = verbose_as)




ℹ️  INFO Train/Test split based on date
Requested train size : 0.7000
Actual train size    : 0.7037
Actual test size     : 0.2963
========== Model: random_forest_fwi ==========
Mean F1 Score: 0.420
Total folds: 8
Average Class Report
              precision    recall  f1-score      support
False          0.652238  0.633199  0.638713  1858.125000
True           0.414861  0.431845  0.419807  1128.875000
accuracy       0.572983  0.572983  0.572983     0.572983
macro avg      0.533549  0.532522  0.529260  2987.000000
weighted avg   0.576584  0.572983  0.570995  2987.000000
========== Model: logistic_regression_fwi ==========
Mean F1 Score: 0.387
Total folds: 8
Average Class Report
              precision    recall  f1-score      support
False          0.686593  0.749428  0.697586  1858.125000
True           0.493854  0.377860  0.387359  1128.875000
accuracy       0.642325  0.642325  0.642325     0.642325
macro avg      0.590223  0.563644  0.542472  2987.000000
weighted avg   0.622561  0.6

In [8]:
import re
from sklearn.decomposition import PCA
pca_default = Pipeline([('scaler', StandardScaler()),
                       ('pca', PCA(n_components = 0.95 ))])
pca_finetuned = Pipeline([('scaler', StandardScaler()),
                       ('pca', PCA(n_components = 0.95 ))])
# Run FWI as this doesnt change across datasets
verbose_as = False
sentinel_feat_cols = [c for c in df_ml.columns if re.search("^feat_", c)]

#df_train, df_test = train_test_temporal_split(df_ml)
df_default  = files['resnet_default_weights']
df_train_default, df_test_default = train_test_temporal_split(df_default)

df_finetuned = files['resnet_layer4_finetuned']
df_train_ft, df_test_ft = train_test_temporal_split(df_finetuned)


#df_sentinel_default = files['resnet_default_weights'].sort_values("date").reset_index(drop=True)
#df_sentinel_finetuned = files['resnet_layer4_finetuned'].sort_values("date").reset_index(drop=True)

# Get Y values 
y_default = df_train_default["fire_lbl_y"]
y_finetuned = df_train_ft["fire_lbl_y"]

# Get data X
X_train_sentinel_default = df_train_default.loc[:, sentinel_feat_cols]
X_train_sentinel_default_pca = pca_default.fit_transform(X_train_sentinel_default)
X_train_sentinel_default_pca = pd.DataFrame(X_train_sentinel_default_pca)
print(f"Total Components selected for default: {pca_default.named_steps['pca'].n_components_}/512")

X_train_sentinel_finetuned = df_train_ft.loc[:, sentinel_feat_cols]
X_train_sentinel_finetuned_pca = pca_finetuned.fit_transform(X_train_sentinel_finetuned)
X_train_sentinel_finetuned_pca = pd.DataFrame(X_train_sentinel_finetuned_pca)

print(f"Total Components selected for Finetuned: {pca_finetuned.named_steps['pca'].n_components_}/512")


rf_sentinel_def     = model_crossvalidation(models['random_forest'], X_train_sentinel_default, y_default, 'rf_sentinel_def', n_splits = 8, verbose = verbose_as)
rf_sentinel_def_pca = model_crossvalidation(models['random_forest'], X_train_sentinel_default_pca, y_default, 'rf_sentinel_def_pca', n_splits = 8, verbose = verbose_as)

rf_sentinel_ft = model_crossvalidation(models['random_forest'], X_train_sentinel_finetuned, y_finetuned, 'rf_sentinel_ft', n_splits = 8, verbose = verbose_as)
rf_sentinel_ft_pca = model_crossvalidation(models['random_forest'], X_train_sentinel_finetuned_pca, y_finetuned, 'rf_sentinel_ft_pca', n_splits = 8, verbose = verbose_as)

        
lr_sentindel_def = model_crossvalidation(models['lr'], X_train_sentinel_default, y_default, 'lr_sentindel_def', n_splits = 8, verbose = verbose_as)
lr_sentindel_def_pca = model_crossvalidation(models['lr'], X_train_sentinel_default_pca, y_default, 'lr_sentindel_def_pca', n_splits = 8, verbose = verbose_as)

lr_sentinel_ft = model_crossvalidation(models['lr'], X_train_sentinel_finetuned, y_finetuned, 'lr_sentinel_ft', n_splits = 8, verbose = verbose_as)
lr_sentinel_ft_pca = model_crossvalidation(models['lr'], X_train_sentinel_finetuned_pca, y_finetuned, 'lr_sentinel_ft_pca', n_splits = 8, verbose = verbose_as)
         
svm_sentinel_def = model_crossvalidation(models['svm'], X_train_sentinel_default, y_default, 'svm_sentinel_def', n_splits = 8, verbose = verbose_as)
svm_sentinel_def_pca = model_crossvalidation(models['svm'], X_train_sentinel_default_pca, y_default, 'svm_sentinel_def_pca', n_splits = 8, verbose = verbose_as)

svm_sentinel_ft  = model_crossvalidation(models['svm'], X_train_sentinel_finetuned, y_finetuned, 'svm_sentinel_ft', n_splits = 8, verbose = verbose_as)
svm_sentinel_ft_pca  = model_crossvalidation(models['svm'], X_train_sentinel_finetuned_pca, y_finetuned, 'svm_sentinel_ft_pca', n_splits = 8, verbose = verbose_as)



ℹ️  INFO Train/Test split based on date
Requested train size : 0.7000
Actual train size    : 0.7037
Actual test size     : 0.2963

ℹ️  INFO Train/Test split based on date
Requested train size : 0.7000
Actual train size    : 0.7037
Actual test size     : 0.2963
Total Components selected for default: 301/512
Total Components selected for Finetuned: 104/512
========== Model: rf_sentinel_def ==========
Mean F1 Score: 0.377
Total folds: 8
Average Class Report
              precision    recall  f1-score      support
False          0.655548  0.749600  0.697997  1858.125000
True           0.446196  0.340296  0.377100  1128.875000
accuracy       0.614998  0.614998  0.614998     0.614998
macro avg      0.550872  0.544948  0.537549  2987.000000
weighted avg   0.590546  0.614998  0.594628  2987.000000
========== Model: rf_sentinel_def_pca ==========
Mean F1 Score: 0.316
Total folds: 8
Average Class Report
              precision    recall  f1-score      support
False          0.641976  0.815068  

In [17]:
import re

# Run FWI as this doesnt change across datasets
verbose_as = False
sentinel_feat_cols = [c for c in df_ml.columns if re.search("^feat_", c)]
hybrid_cols = sentinel_feat_cols + ['fwi_mean']


#df_train, df_test = train_test_temporal_split(df_ml)
# df_hybrid_default = files['resnet_default_weights'].sort_values("date").reset_index(drop=True)
# df_hybrid_finetuned = files['resnet_layer4_finetuned'].sort_values("date").reset_index(drop=True)

# Get Y values 
y_default =   df_train_default["fire_lbl_y"]
y_finetuned = df_train_ft["fire_lbl_y"]

# Get data X
X_train_hybrid_default =   df_train_default.loc[:, hybrid_cols]
X_train_hybrid_finetuned = df_train_ft.loc[:, hybrid_cols]


rf_hybrid_def = model_crossvalidation(models['random_forest'], X_train_hybrid_default, y_default, 'rf_hybrid_def', n_splits = 8, verbose = verbose_as)
rf_hybrid_ft = model_crossvalidation(models['random_forest'], X_train_hybrid_finetuned, y_finetuned, 'rf_hybrid_ft', n_splits = 8, verbose = verbose_as)
        
lr_hybrid_def = model_crossvalidation(models['lr'], X_train_hybrid_default, y_default, 'lr_hybrid_def', n_splits = 8, verbose = verbose_as)
lr_hybrid_ft = model_crossvalidation(models['lr'], X_train_hybrid_finetuned, y_finetuned, 'lr_hybrid_ft', n_splits = 8, verbose = verbose_as)
         
svm_hybrid_def = model_crossvalidation(models['svm'], X_train_hybrid_default, y_default, 'svm_hybrid_def', n_splits = 8, verbose = verbose_as)
svm_hybrid_ft  = model_crossvalidation(models['svm'], X_train_hybrid_finetuned, y_finetuned, 'svm_hybrid_ft', n_splits = 8, verbose = verbose_as)


========== Model: rf_hybrid_def ==========
Mean F1 Score: 0.391
Total folds: 8
========== Model: rf_hybrid_ft ==========
Mean F1 Score: 0.456
Total folds: 8
========== Model: lr_hybrid_def ==========
Mean F1 Score: 0.463
Total folds: 8
========== Model: lr_hybrid_ft ==========
Mean F1 Score: 0.517
Total folds: 8
========== Model: svm_hybrid_def ==========
Mean F1 Score: 0.462
Total folds: 8
========== Model: svm_hybrid_ft ==========
Mean F1 Score: 0.514
Total folds: 8


In [ ]:
from sklearn.ensemble import RandomForestClassifier

verbose_state = True
random_forest_model = RandomForestClassifier(random_state=42, class_weight='balanced')
print("========= FWI ===========")
rf_fwi = model_crossvalidation(   random_forest_model, X_train_fwi, y_train, "rf_fwi_balanced", verbose = verbose_state)

print("========= SENTINEL2 ===========")
rf_sentinel = model_crossvalidation(random_forest_model, X_train_sentinel, y_train, "rf_sentinel_balanced", verbose = verbose_state)
rf_sentinel_ft = model_crossvalidation(random_forest_model, X_train_sentinel_ft, y_train, "rf_sentinel_balanced_finetuned", verbose = verbose_state)

print("========= HYBRID ===========")
rf_hybrid    = model_crossvalidation(random_forest_model, X_train_hybrid,    y_train, "rf_hybrid_balanced", verbose = verbose_state)
rf_hybrid_ft = model_crossvalidation(random_forest_model, X_train_hybrid_ft, y_train, "rf_hybrid_balanced_finetuned", verbose = verbose_state)


In [ ]:
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

svm_model = Pipeline([("scaler", StandardScaler()),
                      ("svm", LinearSVC(C=1.0,
                                        class_weight='balanced',
                                        random_state=42,
                                        max_iter = 1000)) ])

print("========= FWI ===========")
svm_fwi = model_crossvalidation(   svm_model, X_train_fwi, y_train, "svm_fwi_balanced")

print("========= SENTINEL2 ===========")
svm_sentinel = model_crossvalidation(svm_model, X_train_sentinel, y_train, "svm_sentinel_balanced")
svm_sentinel_ft = model_crossvalidation(svm_model, X_train_sentinel_ft, y_train, "svm_sentinel_balanced_finetuned")

print("========= HYBRID ===========")
svm_hybrid    = model_crossvalidation(svm_model, X_train_hybrid,    y_train, "svm_hybrid_balanced")
svm_hybrid_ft = model_crossvalidation(svm_model, X_train_hybrid_ft, y_train, "svm_hybrid_balanced_finetuned")

In [ ]:
from sklearn.linear_model import LogisticRegression

lr_model = Pipeline([("scaler", StandardScaler()),
                      ("lr", LogisticRegression(class_weight='balanced',
                                                random_state=42,
                                                max_iter = 1000)) ])

print("========= FWI ===========")
lr_fwi = model_crossvalidation(   lr_model, X_train_fwi, y_train, "lr_fwi_balanced")

print("========= SENTINEL2 ===========")
lr_sentinel = model_crossvalidation(lr_model, X_train_sentinel, y_train, "lr_sentinel_balanced")
lr_sentinel_ft = model_crossvalidation(lr_model, X_train_sentinel_ft, y_train, "lr_sentinel_balanced_finetuned")

print("========= HYBRID ===========")
lr_hybrid    = model_crossvalidation(lr_model, X_train_hybrid,    y_train, "lr_hybrid_balanced")
lr_hybrid_ft = model_crossvalidation(lr_model, X_train_hybrid_ft, y_train, "lr_hybrid_balanced_finetuned")